# SIMRAS Bridge Risk Model — Live Accuracy Demo

**Purpose:** demonstrate exactly **one trained ML model** and calculate its accuracy live in Google Colab.

**Model artifact:** `artifacts/bridge_nbi/risk.joblib`  
**Model name:** `simras_nbi_bridge_deterioration` — bridge risk classifier  
**Feature version:** `nbi_bridge_state_v1`  
**Data:** official U.S. FHWA National Bridge Inventory (2020–2024), using the same deterministic sampling and bridge-level holdout split as the SIMRAS training code.

### What the judges need to do
**Runtime → Run all.** Everything else is automatic.

The notebook:
1. clones the exact SIMRAS Git commit,
2. verifies the saved model checksum,
3. downloads the matching official NBI source data,
4. rebuilds the deterministic evaluation panel,
5. selects the held-out test bridges,
6. loads **one model only** (`risk.joblib`),
7. calculates live **Accuracy**, Precision, Recall, F1 and ROC-AUC,
8. shows the confusion matrix and sample predictions.

> **Scope note:** the repository itself labels this model `RESEARCH_TRANSFER`. It was trained/evaluated on FHWA NBI bridges and is **not locally validated for Andhra Pradesh**. The accuracy shown below is therefore an FHWA held-out test result, not an Andhra Pradesh field-validation claim.

In [ ]:
# ============================================================
# 1. AUTOMATIC COLAB SETUP
# ============================================================
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/eswarc440-lgtm/SIMRAS.git"
REPO_DIR = Path("/content/SIMRAS")
PINNED_COMMIT = "4685311429dea43580ed32210c5f4f8f61ce02e6"

print("Installing the exact runtime needed by the saved SIMRAS model...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "scikit-learn==1.9.0",
        "pandas>=2.2,<3",
        "numpy>=2.0,<3",
        "joblib>=1.4,<2",
        "matplotlib>=3.9,<4",
    ],
    check=True,
)

if REPO_DIR.exists():
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

print("Cloning SIMRAS...")
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", PINNED_COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "ml"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("\nSETUP PASS")
print("Repository :", REPO_URL)
print("Commit     :", commit)
print("Python     :", sys.version.split()[0])

In [ ]:
# ============================================================
# 2. VERIFY THE ONE MODEL WE WILL DEMONSTRATE
# ============================================================
import hashlib
import json
import joblib
import sklearn

MANIFEST_PATH = REPO_DIR / "artifacts" / "bridge_nbi" / "manifest.json"
RISK_MODEL_PATH = REPO_DIR / "artifacts" / "bridge_nbi" / "risk.joblib"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

EXPECTED_RISK_SHA256 = manifest["artifact_checksums"]["risk.joblib"]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

actual_sha256 = sha256_file(RISK_MODEL_PATH)
if actual_sha256 != EXPECTED_RISK_SHA256:
    raise RuntimeError(
        "Model checksum mismatch. "
        f"Expected {EXPECTED_RISK_SHA256}, got {actual_sha256}"
    )

model = joblib.load(RISK_MODEL_PATH)

print("MODEL VERIFICATION PASS")
print("Model name       :", manifest["model_name"])
print("Model version    :", manifest["version"])
print("Feature version  :", manifest["feature_version"])
print("Stage            :", manifest["stage"])
print("Training scope   :", manifest["training_scope"])
print("Artifact          :", RISK_MODEL_PATH.relative_to(REPO_DIR))
print("SHA256            :", actual_sha256)
print("scikit-learn     :", sklearn.__version__)
print("Pipeline          :", model)
print("\nOnly ONE trained model artifact is loaded: risk.joblib")

In [ ]:
# ============================================================
# 3. AUTOMATICALLY REBUILD THE MATCHING HELD-OUT TEST DATA
# ============================================================
import pandas as pd
import numpy as np

from simras_ml.nbi import (
    FEATURE_COLUMNS,
    add_targets,
    build_panel,
    download_year,
    stable_partition,
)

YEARS = [int(y) for y in manifest["training_years"]]
HORIZON_YEARS = int(manifest["prediction_horizon_years"])
DATA_DIR = Path("/content/simras_nbi_official")
MAX_RECORDS_PER_YEAR = 75_000

print("Downloading/caching official FHWA NBI archives...")
archives = []

for year in YEARS:
    print(f"  {year} ...", end=" ", flush=True)
    archive_path = download_year(
        year,
        DATA_DIR,
        accept_disclaimer=True,
    )
    archives.append((year, archive_path))
    print(f"OK ({archive_path.stat().st_size / 1_000_000:.1f} MB)")

print("\nBuilding the same deterministic bridge-year panel used by SIMRAS...")
panel = build_panel(
    archives,
    max_records_per_year=MAX_RECORDS_PER_YEAR,
)

print("Creating leakage-safe future risk labels...")
labelled = add_targets(
    panel,
    horizon_years=HORIZON_YEARS,
)

# Same risk-data rule and same stable bridge-level split as nbi_train.py.
risk_data = labelled.dropna(subset=["poor_within_horizon"]).copy()
partitions = risk_data["bridge_key"].map(stable_partition)
risk_test = risk_data.loc[partitions >= 85].copy()

if risk_test.empty:
    raise RuntimeError("Held-out risk test partition is empty.")

if risk_test["poor_within_horizon"].nunique() < 2:
    raise RuntimeError("Held-out test partition does not contain both risk classes.")

expected_training_rows = int(manifest["training_rows"])
if len(labelled) != expected_training_rows:
    print(
        "\nNOTE: Rebuilt labelled-row count differs from the stored manifest:",
        f"{len(labelled):,} vs {expected_training_rows:,}.",
        "This can happen if the upstream FHWA archive changed.",
        "The accuracy below is still calculated live on the rebuilt deterministic holdout."
    )

print("\nDATA PREPARATION PASS")
print("Years              :", YEARS)
print("Panel rows          :", f"{len(panel):,}")
print("Labelled rows       :", f"{len(labelled):,}")
print("Risk-evaluable rows :", f"{len(risk_data):,}")
print("Held-out test rows  :", f"{len(risk_test):,}")
print("Held-out bridges    :", f"{risk_test['bridge_key'].nunique():,}")
print("Positive prevalence :", f"{risk_test['poor_within_horizon'].mean() * 100:.2f}%")

In [ ]:
# ============================================================
# 4. RUN THE ONE MODEL AND CALCULATE LIVE ACCURACY
# ============================================================
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

features = list(manifest["features"])
if features != list(FEATURE_COLUMNS):
    raise RuntimeError(
        "Feature contract mismatch between manifest and current SIMRAS code."
    )

X_test = risk_test[features]
y_true = risk_test["poor_within_horizon"].astype(int).to_numpy()

# Exactly one trained classifier is used here.
y_pred = model.predict(X_test).astype(int)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_true, y_pred)
balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)

print("\n" + "=" * 66)
print("          SIMRAS BRIDGE RISK MODEL — LIVE TEST RESULT")
print("=" * 66)
print(f"MODEL ACCURACY       : {accuracy * 100:6.2f}%")
print(f"Balanced Accuracy    : {balanced_accuracy * 100:6.2f}%")
print(f"Precision            : {precision * 100:6.2f}%")
print(f"Recall               : {recall * 100:6.2f}%")
print(f"F1 Score             : {f1 * 100:6.2f}%")
print(f"ROC-AUC              : {roc_auc * 100:6.2f}%")
print(f"Held-out test rows   : {len(y_true):,}")
print("=" * 66)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["No poor-condition event", "Poor-condition event"],
    digits=4,
    zero_division=0,
))

In [ ]:
# ============================================================
# 5. CONFUSION MATRIX + SAMPLE ACTUAL VS PREDICTED
# ============================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(cm)

for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, f"{value:,}", ha="center", va="center")

ax.set_xticks([0, 1], labels=["Predicted 0", "Predicted 1"])
ax.set_yticks([0, 1], labels=["Actual 0", "Actual 1"])
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(f"SIMRAS Bridge Risk — Confusion Matrix\nAccuracy = {accuracy * 100:.2f}%")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

demo = risk_test[
    ["bridge_key", "report_year", "condition_rating"] + features
].copy()

demo["actual_risk"] = y_true
demo["predicted_risk"] = y_pred
demo["risk_probability"] = y_prob

print("\nSample held-out predictions:")
display(
    demo[
        [
            "bridge_key",
            "report_year",
            "condition_rating",
            "actual_risk",
            "predicted_risk",
            "risk_probability",
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# 6. SAVE A JUDGE-FRIENDLY RESULT FILE
# ============================================================
result = {
    "project": "SIMRAS",
    "demonstrated_model": "bridge risk classifier",
    "artifact": "artifacts/bridge_nbi/risk.joblib",
    "model_name": manifest["model_name"],
    "model_version": manifest["version"],
    "feature_version": manifest["feature_version"],
    "stage": manifest["stage"],
    "training_scope": manifest["training_scope"],
    "accuracy": float(accuracy),
    "accuracy_percent": float(accuracy * 100),
    "balanced_accuracy": float(balanced_accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "test_rows": int(len(y_true)),
    "test_bridges": int(risk_test["bridge_key"].nunique()),
    "git_commit": PINNED_COMMIT,
    "model_sha256": actual_sha256,
    "note": (
        "Held-out FHWA NBI research-transfer evaluation; "
        "not Andhra Pradesh local field validation."
    ),
}

RESULT_PATH = Path("/content/SIMRAS_BRIDGE_RISK_ACCURACY_RESULT.json")
RESULT_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")

print("\nFINAL RESULT")
print("-" * 66)
print(f"MODEL ACCURACY: {accuracy * 100:.2f}%")
print(f"Saved report  : {RESULT_PATH}")
print("-" * 66)
print(
    "\nJudge explanation:\n"
    "This number was not hard-coded. Colab rebuilt the deterministic "
    "held-out test set from official FHWA NBI data, loaded the saved "
    "SIMRAS bridge risk classifier, predicted every held-out row, and "
    "calculated accuracy by comparing predicted labels with the actual "
    "future poor-condition labels."
)

## What to say to the judges

**“I am demonstrating one model only: the SIMRAS bridge risk classifier.  
The notebook automatically downloads the official FHWA NBI source data, recreates the same deterministic bridge-level holdout test set, loads the saved `risk.joblib` model, runs predictions, and calculates accuracy live. The displayed percentage is calculated from actual-versus-predicted labels; it is not hard-coded.”**

If asked about scope:

**“This artifact is currently a research-transfer model trained on FHWA NBI data. I use its held-out FHWA accuracy as the reproducible ML demonstration; I do not claim that percentage as local Andhra Pradesh field-validation accuracy.”**